# Shipout Analysis
The goal of this project is to reduce the number of shipouts. Each shipout carries a real cost: the price of a shipping label plus the labor hours spent to pack the order. By establishing a baseline level of shipouts we can identify which strategies effectively reduce the quantity of shipouts

In [1]:
import pandas as pd

# Data Source
RICS Inventory Detail and Stock Status reports are used as the data source for this project. Both reports were restricted to only the footwear class. Although there are shipouts for other product categories, the strategies for reducing the number of shipouts will center around footwear distribution. Inventory detail report was run from 1/1/2026-6/8/2026. The stock status report was run to evaluate inventory levels on 6/9/2026.

In [2]:
inventory_detail = pd.read_csv("InventoryDetail.csv", dtype={"Group": str})
inventory_detail.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 449266 entries, 0 to 449265
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Organization     449266 non-null  object 
 1   Group            0 non-null       object 
 2   Sku              449266 non-null  object 
 3   SkuDescription   449266 non-null  object 
 4   SkuColor         449062 non-null  object 
 5   SkuSupplierCode  449266 non-null  object 
 6   SkuClass         449266 non-null  object 
 7   InventoryStore   449266 non-null  int64  
 8   InventoryDate    449266 non-null  object 
 9   GridColumn       449266 non-null  float64
 10  GridRow          448965 non-null  object 
 11  InventoryType    449266 non-null  object 
 12  Qty              449266 non-null  int64  
 13  LineItemCost     449266 non-null  float64
 14  Comment          218947 non-null  object 
dtypes: float64(2), int64(2), object(11)
memory usage: 51.4+ MB


In [3]:
inventory_detail["InventoryDate"] = pd.to_datetime(inventory_detail["InventoryDate"], format="%m/%d/%Y", errors="coerce")
inventory_detail["InventoryDate"].max(), inventory_detail["InventoryDate"].min()

(Timestamp('2026-06-08 00:00:00'), Timestamp('2026-01-01 00:00:00'))

In [4]:
detail_this_year = inventory_detail[inventory_detail["InventoryDate"]>=pd.to_datetime("01/01/2026", format="%m/%d/%Y")]
detail_this_year["InventoryDate"].max(), detail_this_year["InventoryDate"].min()

(Timestamp('2026-06-08 00:00:00'), Timestamp('2026-01-01 00:00:00'))

In [5]:
detail_this_year["InventoryType"].value_counts()

InventoryType
Sale                   154022
Transfer Out           143169
Transfer In            140308
Sale (POS Return)       11712
Transfer Out Cancel        55
Name: count, dtype: int64

In [6]:
transfers = detail_this_year[(detail_this_year["InventoryType"]=="Transfer In")]
len(transfers)

140308

In [7]:
shipouts = transfers[transfers["Comment"].str.startswith("TO# SHP", na=False)]
shipouts.head()

,Organization,Group,Sku,SkuDescription,SkuColor,SkuSupplierCode,SkuClass,InventoryStore,InventoryDate,GridColumn,GridRow,InventoryType,Qty,LineItemCost,Comment
65,1 - 1 Marathon Sports Norwell,NaN,1011B960-002,GEL-CUMULUS 27,BLACK/CONCRETE,ASIC,MEN'S FOOTWEAR,1,2026-02-20,8.5,D,Transfer In,1,77.0,TO# SHP.30.1.0220-LF
67,1 - 1 Marathon Sports Norwell,NaN,1011B960-004,GEL-CUMULUS 27,BLACK/LUCID YELLOW,ASIC,MEN'S FOOTWEAR,1,2026-01-16,8.0,D,Transfer In,1,77.0,TO# SHP.4.1.0115-PD
114,1 - 1 Marathon Sports Norwell,NaN,1011B974-002,NOVABLAST 5,BLACK/CARRIER GREY,ASIC,MEN'S FOOTWEAR,1,2026-03-13,12.5,D,Transfer In,1,77.0,TO# SHP.13.1.0313-HP
125,1 - 1 Marathon Sports Norwell,NaN,1011B974-002,NOVABLAST 5,BLACK/CARRIER GREY,ASIC,MEN'S FOOTWEAR,1,2026-01-16,11.0,D,Transfer In,1,77.0,TO# SHP.3.1.0114-MS
128,1 - 1 Marathon Sports Norwell,NaN,1011B974-003,NOVABLAST 5,BLACK/BLUE FADE,ASIC,MEN'S FOOTWEAR,1,2026-06-08,10.5,D,Transfer In,1,82.5,TO# SHP.13.1.0608-AG


In [18]:
shipouts_by_store = (
    shipouts
    .groupby("InventoryStore")
    .agg(
        shipout_count=("Qty", "sum")
    )
)

In [19]:
sales = detail_this_year[detail_this_year["InventoryType"]=="Sale"]
sales_by_store = (
    sales
    .groupby("InventoryStore")
    .agg(total_sales=("Qty", "sum"))
)
sales_by_store["total_sales"] = -1*sales_by_store["total_sales"]

In [35]:
# shipouts as a percentage of units sold
combined = pd.merge(sales_by_store, shipouts_by_store, left_index=True, right_index=True)
combined["shipout_percent_of_sales"] = combined["shipout_count"] / combined["total_sales"]

In [36]:
stock_status = pd.read_csv("stock_status.csv")
on_hand_by_store = (
    stock_status
    .groupby("StoreCode")
    .agg(total_on_hand=("OnHand", "sum"))
)

In [37]:
combined = pd.merge(combined, on_hand_by_store, left_index=True, right_index=True)
combined

,total_sales,shipout_count,shipout_percent_of_sales,total_on_hand
1,7790,1166,0.149679,3975
2,8374,1156,0.138046,3698
3,7242,1380,0.190555,4239
4,4834,908,0.187836,2441
5,5595,800,0.142985,3153
6,4088,779,0.190558,2905
7,5420,1221,0.225277,2728
9,514,29,0.056420,816
10,2835,430,0.151675,2562
11,2924,784,0.268126,2112


In [47]:
established_combined = combined[~combined.index.isin([31, 32, 9])].copy()
established_combined

,total_sales,shipout_count,shipout_percent_of_sales,total_on_hand
1,7790,1166,0.149679,3975
2,8374,1156,0.138046,3698
3,7242,1380,0.190555,4239
4,4834,908,0.187836,2441
5,5595,800,0.142985,3153
6,4088,779,0.190558,2905
7,5420,1221,0.225277,2728
10,2835,430,0.151675,2562
11,2924,784,0.268126,2112
12,2469,566,0.229243,1722


In [48]:
established_combined["turnover"] = established_combined["total_sales"] / established_combined["total_on_hand"]
established_combined

,total_sales,shipout_count,shipout_percent_of_sales,total_on_hand,turnover
1,7790,1166,0.149679,3975,1.959748
2,8374,1156,0.138046,3698,2.264467
3,7242,1380,0.190555,4239,1.708422
4,4834,908,0.187836,2441,1.980336
5,5595,800,0.142985,3153,1.774500
6,4088,779,0.190558,2905,1.407229
7,5420,1221,0.225277,2728,1.986804
10,2835,430,0.151675,2562,1.106557
11,2924,784,0.268126,2112,1.384470
12,2469,566,0.229243,1722,1.433798


In [50]:
established_combined["transfer_score"] = established_combined["turnover"]/established_combined["shipout_percent_of_sales"]
established_combined.sort_values(by="transfer_score", ascending=False)
# want a metric to reflect stores that request a lot of shipouts despite having high inventory levels
# unlike sales, would expect shipout requests to be lower for higher inventory
# inventory / number of shipouts

,total_sales,shipout_count,shipout_percent_of_sales,total_on_hand,turnover,transfer_score
28,6240,596,0.095513,3606,1.730449,18.117455
2,8374,1156,0.138046,3698,2.264467,16.403676
1,7790,1166,0.149679,3975,1.959748,13.093002
5,5595,800,0.142985,3153,1.774500,12.410413
4,4834,908,0.187836,2441,1.980336,10.542890
20,3535,570,0.161245,2379,1.485918,9.215301
3,7242,1380,0.190555,4239,1.708422,8.965500
7,5420,1221,0.225277,2728,1.986804,8.819390
17,3574,556,0.155568,2653,1.347154,8.659585
14,3895,791,0.203081,2272,1.714349,8.441704


In [46]:
established_combined["shipout_count"].sum() / established_combined["total_sales"].sum(), established_combined["shipout_count"].sum() / established_combined["total_on_hand"].sum()

(np.float64(0.18306445123913742), np.float64(0.27698558840985527))

In [52]:
established_combined.to_csv("shipout_analysis.csv", index=True)